In [20]:
import pandas as pd 
import numpy as np
from pathlib import Path
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [27]:
# Main paths:
data_p = Path.cwd().parent / 'data' if Path.cwd().name == 'notebooks' else Path.cwd() / 'data'

case1_p = data_p/'case1'
test1_p = case1_p/'test1.csv'
train1_p = case1_p/'train1.csv'

# Main data frames:
test1 = pd.read_csv(test1_p)
train1 = pd.read_csv(train1_p)

# Adding KRPM column:
test1['speed_krpm'] = test1['speed_rpm'] / 1000
train1['speed_krpm'] = train1['speed_rpm'] / 1000

test1 = test1.drop(columns=["pump_id", "operating_state", "speed_rpm"])
train1 = train1.drop(columns=["pump_id", "operating_state", "speed_rpm"])

In [ ]:
# Impementing sub functions:

def compute_sse(y: np.ndarray):
    ''' 
    Args: 
        y: ndarray of real target values
    Return: 
        np.float representing the SSE of y
    '''
    mean_y = np.mean(y)

    return np.sum((y-mean_y)**2)

def find_best_split(df: pd.DataFrame, target_variable, min_samples_leaf=1):
    '''
    Args:
        df: pandas Data Frame containing the features
        y: list containing the target values
    Retrun:
        feature: string of the spliting feature
        threshold: float of the spliting threshold
    '''

    y = df[target_variable].to_numpy()
    df = df.drop(target_variable, axis = "columns")

    best_feature = None
    best_threshold = None
    best_sse = np.inf

    for feature in df.columns:
        
        features = df[feature].to_numpy()
        unique_features = np.unique(features)

        for threshold in unique_features:

            mask_right = threshold < features
            mask_left = ~mask_right

            y_right = y[mask_right]
            y_left = y[mask_left]
            
            if len(y_left) < min_samples_leaf or len(y_right) < min_samples_leaf:
                continue

            sse_yr = compute_sse(y_right)
            sse_yl = compute_sse(y_left)

            sse = sse_yr + sse_yl

            if  best_sse > sse :
                best_feature = feature
                best_threshold = threshold
                best_sse = sse
                
    return best_feature, best_threshold

def partition_training_data(feature, threshold, df):
    '''
    Args:
        best_feature: str name of the spliting feature
        best_threshold: str name of the spliting threshold
        df: dataframe of the data you want to split
    Return:
        data_left: dataframe that respects the following condition feature <= threshold
        data_right: dataframe that respects the following condition feature > threshold
    '''

    data_left = df[df[feature] <= threshold]
    data_right = df[df[feature] > threshold]

    return data_left, data_right
    

def build_tree(df, target_variable, depth=0, max_depth=None, min_samples_split=2, min_samples_leaf=1):
    '''
    Args:
        df: pandas DataFrame containing the features and target values
        target_variable: str name of the target column
        depth: int current depth of the tree
        max_depth: int maximum depth of the tree
        min_samples_split: int minimum number of samples required to split a node
        min_samples_leaf: int minimum number of samples required in each leaf
    Return:
        tree: dict representing the regression tree
    '''

    if (max_depth is not None and depth >= max_depth) or len(df) < min_samples_split:
        return {"leaf": df[target_variable].mean()}

    best_feature, best_threshold = find_best_split(df, target_variable, min_samples_leaf)

    if best_feature is None:
        return {"leaf": df[target_variable].mean()}

    data_left, data_right = partition_training_data(best_feature, best_threshold, df)

    left_branch = build_tree(data_left, target_variable, depth + 1, max_depth, min_samples_split, min_samples_leaf)
    right_branch = build_tree(data_right, target_variable, depth + 1, max_depth, min_samples_split, min_samples_leaf)

    return {"node": (best_feature, best_threshold, left_branch, right_branch)}

def predict(values: dict|pd.Series, tree: dict):
    '''
    Args:
        values: dict or pandas Series containing the feature values
        tree: dict representing the regression tree
    Return:
        float representing the predicted target value
    '''

    if "leaf" in tree:
        return tree["leaf"]
    elif values[tree["node"][0]] <= tree["node"][1]:
        return predict(values, tree["node"][2])
    else:
        return predict(values, tree["node"][3])
    
def predict_dataset(df, tree):
    '''
    Args:
        df: pandas DataFrame containing the feature values
        tree: dict representing the regression tree
    Return:
        results: list containing the predicted target values
    '''
    results = []

    for _, values in df.iterrows():
        results.append(predict(values,tree))
    
    return results

In [ ]:
def evaluate_model(y_true, y_pred):
    '''
    Args:
        y_true: array-like containing the real target values
        y_pred: array-like containing the predicted target values
    Return:
        dict containing the MSE, RMSE, MAE and R2 evaluation metrics
    '''
    return {
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

In [42]:
target_variable = "flow_m3_h"

max_depth = 5
min_samples_split = 4
min_samples_leaf = 2

X_train = train1.drop(columns=[target_variable])
y_train = train1[target_variable]

X_test = test1.drop(columns=[target_variable])
y_test = test1[target_variable]

my_tree = build_tree(
    train1,
    target_variable,
    max_depth=max_depth,
    min_samples_split=min_samples_split,
    min_samples_leaf=min_samples_leaf
)

my_predictions = predict_dataset(X_test, my_tree)

sk_tree = DecisionTreeRegressor(
    max_depth=max_depth,
    min_samples_split=min_samples_split,
    min_samples_leaf=min_samples_leaf,
    random_state=42
)

sk_tree.fit(X_train, y_train)
sk_predictions = sk_tree.predict(X_test)

my_metrics = evaluate_model(y_test, my_predictions)
sk_metrics = evaluate_model(y_test, sk_predictions)

comparison = pd.DataFrame({
    "My Decision Tree": my_metrics,
    "Scikit-learn": sk_metrics
})

comparison

,My Decision Tree,Scikit-learn
MSE,79.066772,80.355139
RMSE,8.891950,8.964103
MAE,6.450620,6.553946
R2,0.457507,0.448667


In [44]:
my_train_predictions = predict_dataset(X_train, my_tree)
sk_train_predictions = sk_tree.predict(X_train)

my_train_metrics = evaluate_model(y_train, my_train_predictions)
sk_train_metrics = evaluate_model(y_train, sk_train_predictions)

comparison = pd.DataFrame({
    "My Tree (Train)": my_train_metrics,
    "My Tree (Test)": my_metrics,
    "Scikit-learn (Train)": sk_train_metrics,
    "Scikit-learn (Test)": sk_metrics
})

comparison

,My Tree (Train),My Tree (Test),Scikit-learn (Train),Scikit-learn (Test)
MSE,37.116354,79.066772,37.116354,80.355139
RMSE,6.092319,8.891950,6.092319,8.964103
MAE,4.699729,6.450620,4.699729,6.553946
R2,0.782157,0.457507,0.782157,0.448667
